# CFO — ¿Por qué cambió el MRR?

Evaluamos el modelo actual y una política de descuentos para distinguir servicios, tarifas y descuentos, y decidir si el ingreso adicional compensa la reducción concedida.

**Método:** enfocar → comprobar → localizar → explicar → actuar → evaluar.

<details><summary>Ver las seis fases del método</summary>
<ol>
<li><strong>Enfocar:</strong> responder al CFO y apoyar la decisión sobre descuentos.</li>
<li><strong>Comprobar:</strong> revisar qué permiten los datos y comparar el ingreso con y sin política.</li>
<li><strong>Localizar:</strong> pasar del negocio completo a los 50 clientes con descuento.</li>
<li><strong>Explicar:</strong> separar servicios, tarifas y descuentos; conciliar el resultado por períodos.</li>
<li><strong>Actuar:</strong> proponer un ajuste de monto o duración, con responsables.</li>
<li><strong>Evaluar:</strong> comprobar el modelo y definir cómo medir la oferta ajustada en un piloto.</li>
</ol>

</details>

## 1. Qué responde el modelo actual y qué debemos agregar

**Cliente + mes + monto pagado permite medir pagos y sus variaciones; no explica por qué cambió el MRR.** Tampoco confirma churn ni la vida completa del cliente: ausencia de pago no implica baja y el primer pago observado puede no ser el primero histórico.

**Modelo propuesto:** servicios, cantidades y tarifas para calcular el valor recurrente **bruto**; descuento aplicado y fechas para obtener el **neto**. El cobro se registra aparte: puede diferir por atrasos u otros cargos.

**Inicio y fin del descuento:** si solo cambia el neto, clasificamos el movimiento por descuento. Si cambia el bruto, distinguimos servicios y tarifas. Al vencer, el aumento neto no representa una nueva ampliación de servicios.

<details><summary>Ver las tres preguntas del reto y precisiones del modelo</summary>
<ol>
<li>¿Qué puede y qué no puede responder el modelo actual (cliente + mes + monto pagado)?</li>
<li>¿Cómo separarías el valor de la suscripción del precio efectivamente pagado? ¿Qué campos, tablas o definiciones agregarías?</li>
<li>¿Cómo clasificarías el inicio y el fin de un descuento para que no se confundan con contracción o expansión reales?</li>
</ol>
<p><strong>Cobertura:</strong> Transactions.csv contiene ID, month y amount; Industry.csv añade industria. Agregamos por cliente/mes. Podemos medir pagos, continuidad y primer/último pago observados; sin contratos y estados no confirmamos inicio, baja ni retención contractual.</p>
<p><strong>Definiciones:</strong> bruto = cantidad × tarifa; neto = bruto − descuento aplicado. El monto pagado es el cobro efectivo. En esta demostración: pagado = neto − pendiente + cargo no recurrente; pendiente negativo representa cobro recuperado. El porcentaje equivalente del descuento fijo es informativo, no se aplica dos veces.</p>
<p><strong>Vigencias:</strong> registramos inicio y fin por servicio, tarifa y descuento. Una prórroga conserva el historial anterior. Al vencer el descuento, bruto y neto coinciden si no existe otro descuento aplicado.</p>
<p><strong>Historia del cliente:</strong> proponemos fecha del primer pago histórico y cobertura del historial, separadas del inicio contractual. Una fecha desconocida no se sustituye por el primer pago del extracto. Para medir vida completa necesitamos también la baja definitiva; los activos solo tienen duración observada hasta el corte.</p>

</details>

<details><summary>Respuesta a las preguntas del CFO</summary>
<ol>
<li>Si un cliente pagaba 100 y ahora paga 80, ¿contrajo 20 o recibió un descuento?</li>
<li>¿Qué pasa si su suscripción creció de 100 a 130, pero tiene un descuento de 30 y sigue pagando 100?</li>
<li>¿Y cuando desaparezca el descuento, eso cuenta como expansión?</li>
</ol>
<p><strong>Con esa información, podemos responder:</strong></p>
<ol>
<li>Si el bruto baja de 100 a 80 sin descuento, hubo contracción del valor bruto y del MRR neto. Si el bruto permanece en 100 y aplicamos un descuento de 20, hay contracción del MRR neto por descuento, sin disminución del bruto.</li>
<li>Si el bruto aumenta de 100 a 130 con descuento de 30, aumenta el valor bruto, pero el neto permanece en 100. El aumento puede deberse a más servicios o a una mayor tarifa; distinguimos ambas causas.</li>
<li>Al vencer el descuento, el neto aumenta a 130: hay expansión del MRR neto por fin del descuento, sin nuevo aumento del bruto ni servicios adicionales.</li>
</ol>
<p><strong>Criterio comercial:</strong> un descuento sin aumento del bruto puede buscar retención o reactivación. Toda política debe tener un objetivo, presupuesto y proyección financiera para evaluar si el beneficio esperado compensa el descuento.</p>

</details>

## 2. Datos sintéticos para demostrar el modelo

Para cubrir la información faltante creamos datos sintéticos y construimos las métricas. **Unidades monetarias (UM)**: unidad de los importes, sin representar una moneda específica.

Estas dos muestras muestran la separación necesaria; las demás tablas y sus columnas están disponibles al desplegar el detalle.

In [1]:
import sys, json
from pathlib import Path
from IPython.display import HTML, Markdown, display
from html import escape
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/cfo_model.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from scripts.cfo_model import CONFIG
from scripts.cfo_audit import check_persisted
from scripts.cfo_notebook_views import NotebookData, table_html, numero_es
from scripts import cfo_present
ESCENARIO = 'principal'
datos = NotebookData(ROOT, ESCENARIO)
FIG_CONFIG = {'displaylogo': False, 'responsive': True}
display(HTML('<style>.scroll{overflow-x:auto}.scroll table{border-collapse:collapse}.scroll td,.scroll th{padding:8px;border-bottom:1px solid #ddd;text-align:left;white-space:nowrap}details{margin:12px 0}summary{cursor:pointer}</style>'))
def mostrar_conclusion(texto):
    display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;border-radius:8px;">'+escape(texto)+'</div>'))
def mostrar_detalle(titulo,frame):
    display(HTML('<details><summary>'+escape(titulo)+'</summary>'+table_html(frame)+'</details>'))

### Valor recurrente: bruto, descuento y neto

Seguimos un servicio del mismo cliente antes, durante y al vencer el descuento.

In [2]:
display(HTML(datos.preview('componentes_mes')))

### Cobro: separado del valor recurrente

El atraso de julio y su recuperación en agosto cambian el cobro, sin cambiar el MRR.

In [3]:
display(HTML(datos.preview('pagos')))

In [4]:
detalle_modelo = (('<h3 id="A.-Clientes:-a-qui%C3%A9n-aplicamos-la-pol%C3%ADtica">A. Clientes: a qui&#233;n aplicamos la pol&#237;tica<a class="anchor-link" href="#A.-Clientes:-a-qui%C3%A9n-aplicamos-la-pol%C3%ADtica">&#182;</a></h3><p>Una fila representa un <strong>cliente</strong>. Guarda si es existente, nuevo o reactivado y si es elegible para el descuento. Permite fijar la población objetivo sin confundir más contratos con más clientes.</p>\n' + datos.preview('clientes')) +
('<h3 id="B.-Suscripciones:-qu%C3%A9-contratos-tiene-cada-cliente">B. Suscripciones: qu&#233; contratos tiene cada cliente<a class="anchor-link" href="#B.-Suscripciones:-qu%C3%A9-contratos-tiene-cada-cliente">&#182;</a></h3><p>Una fila representa una <strong>suscripción recurrente</strong>. Un cliente puede tener varias. Inicio y fin permiten conocer qué contrato está vigente; historia previa distingue una reactivación de un primer contrato.</p>\n' + datos.preview('suscripciones')) +
('<h3 id="C.-Componentes:-qu%C3%A9-servicios-contrat%C3%B3">C. Componentes: qu&#233; servicios contrat&#243;<a class="anchor-link" href="#C.-Componentes:-qu%C3%A9-servicios-contrat%C3%B3">&#182;</a></h3><p>Una fila representa un <strong>servicio o partida de una suscripción</strong>, como asientos o un módulo. Su identificador conecta las cantidades del mes con el precio y descuento que corresponden. Más servicios solo es expansión monetaria si aumenta su valor recurrente.</p>\n' + datos.preview('componentes')) +
('<h3 id="D.-Precios:-cu%C3%A1nto-vale-cada-unidad">D. Precios: cu&#225;nto vale cada unidad<a class="anchor-link" href="#D.-Precios:-cu%C3%A1nto-vale-cada-unidad">&#182;</a></h3><p>Una fila representa una <strong>versión de tarifa</strong>. Si cambia el precio sin cambiar el servicio, se registra otra vigencia. Así separamos ampliación de servicios de aumento de tarifa. El inicio es inclusivo y el fin exclusivo.</p>\n' + datos.preview('precios')) +
('<h3 id="E.-Descuentos:-qu%C3%A9-reducci%C3%B3n-comercial-se-aplica">E. Descuentos: qu&#233; reducci&#243;n comercial se aplica<a class="anchor-link" href="#E.-Descuentos:-qu%C3%A9-reducci%C3%B3n-comercial-se-aplica">&#182;</a></h3><p>Una fila representa un <strong>descuento temporal sobre un componente</strong>. Monto, inicio y fin determinan cuánto se reduce el ingreso y cuándo vuelve al valor bruto. El porcentaje equivalente es informativo: no se aplica además del monto fijo.</p>\n' + datos.preview('descuentos')) +
('<h3 id="F.-Estados:-qu%C3%A9-suscripciones-permanecen-activas">F. Estados: qu&#233; suscripciones permanecen activas<a class="anchor-link" href="#F.-Estados:-qu%C3%A9-suscripciones-permanecen-activas">&#182;</a></h3><p>Una fila representa una <strong>suscripción en un mes</strong>. Activo/baja/inactivo permite medir permanencia y churn. No recibir un pago no significa automáticamente que el contrato terminó.</p>\n' + datos.preview('estados')) +
('<h3 id="I.-Movimientos:-por-qu%C3%A9-cambi%C3%B3-el-MRR-del-cliente">I. Movimientos: por qu&#233; cambi&#243; el MRR del cliente<a class="anchor-link" href="#I.-Movimientos:-por-qu%C3%A9-cambi%C3%B3-el-MRR-del-cliente">&#182;</a></h3><p>Una fila representa un <strong>cliente en un mes</strong>, después de sumar sus suscripciones. El puente separa servicio/uso, tarifa y cambio de descuento; primera activación, baja y retorno se identifican por estado. Es una salida del modelo, no otra fuente independiente.</p>\n' + datos.preview('movimientos')))
display(HTML('<details><summary>Ver las demás tablas, columnas y relaciones del modelo</summary>'+detalle_modelo+'<h3 id="Relaciones-del-modelo-propuesto">Relaciones del modelo propuesto<a class="anchor-link" href="#Relaciones-del-modelo-propuesto">&#182;</a></h3><p>Cliente → suscripciones → componentes → precios/descuentos. Agregamos componentes a suscripción y cliente por mes antes de unir resultados; unir todo directamente multiplicaría importes.</p>\n'+'</details>'))
controles = check_persisted(ROOT, ESCENARIO)
display(HTML('<details><summary>Ver controles ejecutados: archivos, relaciones e importes</summary><p>'+escape(f"{controles['tablas']} tablas conciliadas; {controles['movimientos']} movimientos reconstruidos; {controles['clientes_foco']} clientes en el foco. CSV, SQLite, vigencias, pagos y métricas concilian.")+'</p></details>'))

## 3. ¿La política mejora el ingreso recurrente neto?

**Escenario sintético:** 50 clientes amplían sus servicios de 100 a 110 UM; reciben 40 UM de descuento mensual durante julio–septiembre. Su neto pasa a 70 y, desde octubre, a 110. Sin política conservan sus servicios de 100 UM.

Comparamos los mismos clientes. Incorporaciones, reactivaciones, bajas y tarifas ajenas a la política se mantienen iguales en ambas alternativas; por eso el total sin política también varía.

### 3.1. ¿Cómo cambia el MRR frente a no aplicar la política?

La gráfica muestra **todo el negocio**, enero 2025–marzo 2026: seis meses previos, tres de descuento y seis posteriores para observar la recuperación.

<details><summary>Ver métrica y datos de 3.1</summary>
<p><strong>Qué medimos:</strong> evolución del bruto, descuento y neto. <strong>Métrica:</strong> MRR neto = suma(bruto − descuento). <strong>Datos:</strong> componentes por mes, agregados a cliente y total; mensual.csv.</p>

</details>

In [5]:
cfo_present.figuras_principales(ESCENARIO)['10_evolucion_principal'].show(renderer='plotly_mimetype',config=FIG_CONFIG)
mostrar_conclusion(datos.monthly_conclusion())
mostrar_detalle('Ver valores de la gráfica', datos.monthly[['mes','grupo','bruto','descuentos','neto']].assign(bruto=lambda d:d.bruto/100,descuentos=lambda d:d.descuentos/100,neto=lambda d:d.neto/100))

mes,grupo,bruto,descuentos,neto
2025-01-01,sin_descuento,10.200,0,10.200
2025-02-01,sin_descuento,10.200,0,10.200
2025-03-01,sin_descuento,10.200,0,10.200
2025-04-01,sin_descuento,10.200,0,10.200
2025-05-01,sin_descuento,10.200,0,10.200
2025-06-01,sin_descuento,10.200,0,10.200
2025-07-01,sin_descuento,10.700,0,10.700
2025-08-01,sin_descuento,10.700,0,10.700
2025-09-01,sin_descuento,11.200,0,11.200
2025-10-01,sin_descuento,10.700,0,10.700


### 3.2. ¿Y cuánto revenue estamos dejando de capturar por decisiones comerciales?

**¿El aumento mensual compensa los descuentos desde julio?**

Seguimos a los **50 clientes con descuento** durante julio 2025–marzo 2026. La tabla compara todo el negocio; la gráfica muestra esos 50 clientes, que explican toda la diferencia.

Las líneas son acumuladas: **saldo = ingreso por ampliación − descuentos aplicados**. Cero indica equilibrio; un saldo negativo significa menos ingreso que sin política.

In [6]:
periodo = datos.monthly[datos.monthly.mes.ge(CONFIG['politica_inicio'])].copy()
comparacion = periodo.groupby('grupo').agg(meses=('mes', 'nunique'), acumulado=('neto', 'sum'))
assert comparacion.meses.nunique() == 1
comparacion['promedio'] = comparacion.acumulado / comparacion.meses
con = comparacion.loc['con_descuento']
sin = comparacion.loc['sin_descuento']
resumen = pd.DataFrame([
    {'Medida': 'Ingreso neto acumulado (UM)', 'Con política': con.acumulado/100, 'Sin política': sin.acumulado/100, 'Diferencia': (con.acumulado-sin.acumulado)/100},
    {'Medida': 'Promedio mensual (UM)', 'Con política': con.promedio/100, 'Sin política': sin.promedio/100, 'Diferencia': (con.promedio-sin.promedio)/100}
])
display(HTML(table_html(resumen)))

Medida,Con política,Sin política,Diferencia
Ingreso neto acumulado (UM),92.700,94.200,-1.500
Promedio mensual (UM),10.300,"10.466,67","-166,67"


In [7]:
fases_elegibles = datos.phases[datos.phases.poblacion.eq('Elegibles')].set_index('fase')
durante = fases_elegibles.loc['durante', 'diferencia_neta'] / 100
despues = fases_elegibles.loc['posterior', 'diferencia_neta'] / 100
saldo = durante + despues
cfo_present.figuras_principales(ESCENARIO)['12_recuperacion_foco'].show(renderer='plotly_mimetype',config=FIG_CONFIG)
explicacion = f"Durante el descuento, los 50 clientes generan {numero_es(abs(durante))} UM {'menos' if durante < 0 else 'más'} que sin política; después, {numero_es(abs(despues))} UM {'más' if despues > 0 else 'menos'}."
mostrar_conclusion(datos.recovery_conclusion() + ' ' + explicacion + ' El resultado cubre nueve meses; evaluar el valor del cliente durante toda su relación (LTV) requiere comparar permanencia e ingresos con y sin política.')
mostrar_detalle('Ver acumulados por mes (UM)',datos.focus.loc[datos.focus.mes.ge(CONFIG['politica_inicio']), ['mes','mes_relativo','diferencia_acumulada','aumento_bruto_acumulado','descuento_acumulado']].assign(diferencia_acumulada=lambda d:d.diferencia_acumulada/100,aumento_bruto_acumulado=lambda d:d.aumento_bruto_acumulado/100,descuento_acumulado=lambda d:d.descuento_acumulado/100))

mes,mes_relativo,diferencia_acumulada,aumento_bruto_acumulado,descuento_acumulado
2025-07-01,0,-1.500,500,2.000
2025-08-01,1,-3.000,1.000,4.000
2025-09-01,2,-4.500,1.500,6.000
2025-10-01,3,-4.000,2.000,6.000
2025-11-01,4,-3.500,2.500,6.000
2025-12-01,5,-3.000,3.000,6.000
2026-01-01,6,-2.500,3.500,6.000
2026-02-01,7,-2.000,4.000,6.000
2026-03-01,8,-1.500,4.500,6.000


## 4. Conclusión, decisión, acción y evaluación

In [8]:
from re import sub
cierre = ''.join('<p>'+sub(r'\*\*(.*?)\*\*', r'<strong>\1</strong>', escape(parrafo))+'</p>' for parrafo in datos.action().split('\n\n'))
display(HTML('<div style="font-family:Arial,sans-serif;font-size:20px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;border-radius:8px;">'+cierre+'</div>'))
display(HTML(table_html(datos.sensitivity_adjustment())))

Condición,Descuento mensual (UM),Meses de descuento,Diferencia acumulada (UM)
Actual,40,3,-1.500
"Umbral, misma ampliación",30,3,0
"Un mes menos, misma ampliación",40,2,500


<details><summary>Supuestos, información faltante y uso de IA</summary>
<ul>
<li><strong>Preguntas priorizadas:</strong> por qué cambia el MRR y si el ingreso adicional compensa los descuentos; las preguntas originales están conservadas arriba.</li>
<li><strong>Supuestos:</strong> ampliación 100→110, descuento 40×3 y permanencia idéntica entre alternativas. El aumento de servicios es un supuesto del escenario; esta comparación no estima un efecto causal. Los ajustes de la tabla conservan ampliación y permanencia.</li>
<li><strong>Información faltante:</strong> servicios, cantidades, tarifas, descuentos y estados contractuales; historia completa para vida del cliente y costos para margen. Los seis meses posteriores permiten observar recuperación, no toda la relación.</li>
<li><strong>Cambio del modelo:</strong> separar servicio, precio, descuento, estado y cobro; agregar al cliente por mes sin multiplicar importes al unir tablas.</li>
<li><strong>IA y calidad:</strong> el candidato priorizó las preguntas, eligió el escenario y revisó la presentación; la IA apoyó código, datos sintéticos y redacción. Se validan claves, vigencias, cálculos, conciliación CSV/SQLite y conclusiones calculadas. El piloto es una propuesta, no una acción ejecutada.</li>
</ul>

</details>

<details><summary>Qué análisis ampliarían la decisión</summary>
<p>Podemos profundizar con preguntas como:</p>
<ul>
<li><strong>¿Mejora la permanencia o la retención de ingresos?</strong> Comparar bajas e ingreso de los mismos clientes con y sin política.</li>
<li><strong>¿Mejora el valor del cliente a largo plazo?</strong> Comparar ingreso y permanencia durante toda la relación, más allá de nueve meses.</li>
<li><strong>¿Genera adopción de nuevos servicios?</strong> Comprobar si los servicios adicionales se mantienen después del descuento.</li>
<li><strong>¿Mejora la rentabilidad?</strong> Incorporar los costos para evaluar el margen, además del ingreso.</li>
</ul>
<p>El resultado actual corresponde a ingresos durante nueve meses, no a utilidad ni al valor completo del cliente.</p>

</details>

[Definiciones, controles y reproducción del análisis](../docs/datos/modelo-cfo-demo.md).